# Theme 2: RAG Application, Starter Kit

This starter kit contains a complete, working **Retrieval-Augmented Generation (RAG)** application using **Gemma 4**, Google's open-weight model. Your team's task is to adapt and extend it for the problem statement you selected (2A or 2B).

**How RAG works:**

```
 Document ──► 1. split into chunks ──► 2. convert each chunk into an embedding (a list of numbers)
                                                           │
 Question ──► 3. retrieve the chunks whose embeddings are closest to the question's embedding
                                                           │
              4. give ONLY those chunks to Gemma ──► answer with page citations
```

**Models used (both open-weight):**
- `all-MiniLM-L6-v2` (sentence-transformers) creates the embeddings. It runs inside this notebook.
- `gemma-4-26b-a4b-it` (Google) writes the answers. It is accessed through the free Google AI Studio API.

**No GPU is required.** Use the default Colab runtime (CPU).

### Obtaining a free Google AI Studio API key (one per team)
1. Go to **https://aistudio.google.com/apikey** and sign in with a Google account.
2. Select **Create API key** and copy the key.
3. In Colab, select the **key icon** in the left sidebar (**Secrets**).
4. Select **Add new secret**. Set **Name** to `GOOGLE_API_KEY` and **Value** to your key. Enable **Notebook access**.

> **Important:** do not paste the key directly into a code cell. Anyone the notebook is shared with would be able to see it.
>
> **No key yet:** the notebook still runs. The retrieval stages work normally, and the application shows the passages it found in place of a written answer.

### How to use this notebook
- Run the cells **in order, from top to bottom**, the first time (select a cell and press **Shift + Enter**).
- Comments marked **STUDENT TODO** show the intended places to make changes.
- After changing a cell, re-run that cell and any cells below it that depend on it.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

## Step 0: Installation (run once)

In [2]:
# Installs the required libraries. This takes about one minute.
!pip install -q pypdf sentence-transformers "gradio>=6,<7" google-genai

## Step 1: Settings

These values control how documents are divided, searched and answered. They are the main parameters to experiment with.

In [3]:
# STUDENT TODO: the Gemma 4 model that writes the answers.
#   "gemma-4-26b-a4b-it"  Mixture of Experts, 26B total / about 4B active parameters. Fast. (default)
#   "gemma-4-31b-it"      Dense, 31B parameters. More capable, slower.
GEMMA_MODEL = "gemma-4-26b-a4b-it"

# STUDENT TODO: the number of words in each chunk.
#   Small chunks (50)   more precise matches, but may lose context
#   Large chunks (300)  more context, but less precise and longer prompts
CHUNK_SIZE = 120

# STUDENT TODO: the number of words shared by neighbouring chunks, so sentences are not split.
CHUNK_OVERLAP = 30

# STUDENT TODO: the number of chunks given to Gemma for each question.
TOP_K = 3

# STUDENT TODO: the rules Gemma follows when answering. The rule to use only the context prevents invented answers.
ANSWER_RULES = """You answer questions about a document using ONLY the context passages provided.
- Each passage begins with its page number, for example [Page 2].
- After each fact you use, cite the page in brackets, for example (p. 2).
- If the answer is not in the context, reply exactly: "The answer could not be found in the document."
- Keep answers concise: 2 to 4 sentences."""

TEMPERATURE = 0.2       # a low value keeps answers consistent and close to the document

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # STUDENT TODO (optional): try "BAAI/bge-small-en-v1.5"
print("Settings configured.")

Settings configured.


## Step 2: Connect to Gemma and load the embedding model

In [4]:
import os
from google import genai
from google.genai import types, errors


def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


GOOGLE_API_KEY = _get_secret("GOOGLE_API_KEY")
DEMO_MODE = not GOOGLE_API_KEY
client = None if DEMO_MODE else genai.Client(api_key=GOOGLE_API_KEY)


def _friendly_error(e):
    code = getattr(e, "code", None)
    if code in (400, 401, 403) and "key" in str(e).lower():
        return "Error: the GOOGLE_API_KEY was rejected. Check that the complete key is saved in Colab Secrets, then re-run this cell."
    if code == 429:
        return "Error: the free-tier usage limit has been reached. Wait about one minute and try again."
    if code == 404:
        return f"Error: the model '{GEMMA_MODEL}' was not found. Check the GEMMA_MODEL setting."
    return f"Error: the request to Gemma failed ({e})."


def generate(user_text, system_text=None, temperature=TEMPERATURE, demo_reply="(Demonstration mode response)"):
    """Sends one message to Gemma and returns its reply as text.
    In demonstration mode, returns `demo_reply`. If the request fails, returns a message beginning with "Error:"."""
    if DEMO_MODE:
        return demo_reply
    try:
        response = client.models.generate_content(
            model = GEMMA_MODEL,
            contents = user_text,
            config = types.GenerateContentConfig(
                system_instruction = system_text,
                temperature = temperature,
                thinking_config = types.ThinkingConfig(thinking_level="minimal"),  # faster replies
            ),
        )
        return (response.text or "").strip()
    except errors.APIError as e:
        return _friendly_error(e)
    except Exception as e:
        return f"Error: {e}"


if DEMO_MODE:
    print("Demonstration mode: no GOOGLE_API_KEY was found. Retrieval will work; answers will show the passages found.")
else:
    print(f"Connected to Google AI Studio. Model: {GEMMA_MODEL}")
    print(generate("In one sentence, what is an open-weight AI model?"))

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_NAME)   # runs on the CPU
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")

Connected to Google AI Studio. Model: gemma-4-26b-a4b-it
An open-weight AI model is a machine learning model where the trained parameters (weights) are made publicly available, allowing developers to run, study, and fine-tune the model on their own hardware.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


## Step 3: Sample document
A short, fictional course syllabus is provided so the application works immediately. Each item in the list is one page. Your own PDF can be uploaded in the application in Step 5.

In [5]:
# STUDENT TODO: replace with text relevant to your problem statement, or upload a PDF in the application instead.
SAMPLE_PAGES = [
    """CS 101: Introduction to Programming (Sample Syllabus). Instructor: Dr. A. Rao. Email: arao@example.edu.
    Lectures are held Monday and Wednesday, 10:00 to 11:30 AM, in Room B-204. Labs are on Friday from 2:00 to 4:00 PM
    in the Computer Lab (Room C-110). Office hours: Tuesday 3:00 to 5:00 PM, or by appointment. This course introduces
    programming using Python. No prior coding experience is required.""",

    """Grading. Weekly lab exercises: 20%. Two mini-projects: 30% (15% each). Midterm exam: 20%. Final exam: 25%.
    Class participation: 5%. A score of 90% or above earns an A, 80 to 89% a B, 70 to 79% a C, and 60 to 69% a D.
    Late submissions lose 10% per day, up to a maximum of 3 days, after which they receive zero. Each student has
    two grace days per semester that may be used to submit any lab late without penalty.""",

    """Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and loops. Week 5: functions.
    Week 6: lists and dictionaries. Week 7: Midterm exam (in class, closed book). Weeks 8-9: files and error handling.
    Weeks 10-11: an introduction to APIs and working with data. Weeks 12-13: mini-project 2, a team project of up to
    3 students. Week 14: review. The final exam is a 2-hour written exam during the examination period.""",

    """Policies. Attendance: students must attend at least 75% of lectures to be eligible for the final exam.
    Academic integrity: students may discuss ideas with classmates, but all submitted code must be their own. AI
    tools may be used for learning and debugging, but their use must be stated in a comment at the top of the file.
    Laptops: bring a laptop to every lab. Students without a laptop may reserve a machine in the Computer Lab.
    Textbook: "Automate the Boring Stuff with Python" (available free online).""",
]
print(f"Sample document contains {len(SAMPLE_PAGES)} pages.")

Sample document contains 4 pages.


## Step 4: The RAG pipeline
Each function is one stage of the pipeline. This is the core of the project and the main code to extend.

In [10]:
import numpy as np
from pypdf import PdfReader

knowledge_base = {"name": None, "chunks": [], "vectors": None}  # populated by build_knowledge_base()


def read_pdf(path):
    """Returns a list of page texts from a PDF file."""
    return [page.extract_text() or "" for page in PdfReader(path).pages]


def split_into_chunks(pages):
    """STAGE 1: split each page into overlapping chunks of about CHUNK_SIZE words."""
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for page_number, text in enumerate(pages, start=1):
        words = text.split()
        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]
            if len(chunk_words) > 5:  # skip very short fragments
                chunks.append({"page": page_number, "text": " ".join(chunk_words)})
            if start + CHUNK_SIZE >= len(words):
                break
    return chunks


def build_knowledge_base(pages, name):
    """STAGE 2: convert each chunk into an embedding."""
    chunks = split_into_chunks(pages)
    if not chunks:
        raise ValueError("No text was found. The PDF may contain scanned images only.")
    vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
    knowledge_base.update(name=name, chunks=chunks, vectors=vectors)
    return f"Loaded **{name}**: {len(pages)} pages, {len(chunks)} chunks."


def retrieve(question, top_k=TOP_K):
    """STAGE 3: find the chunks whose meaning is closest to the question."""
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = knowledge_base["vectors"] @ question_vector   # 1.0 = identical meaning
    best = np.argsort(similarities)[::-1][:top_k]
    return [{**knowledge_base["chunks"][i], "score": float(similarities[i])} for i in best]


def answer(question):
    """STAGE 4: give the retrieved chunks to Gemma and ask it to answer using only them."""
    found = retrieve(question)
    context = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in found)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    demo = (f"**Demonstration mode:** no API key is set, so Gemma did not write an answer. Retrieval completed; "
            f"the most relevant passage is on **page {found[0]['page']}**:\n\n> {found[0]['text'][:400]}...")
    reply = generate(prompt, system_text=ANSWER_RULES, demo_reply=demo)
    return reply, found


# Quick test with the sample document.
print(build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus"))
test_question = "What happens if I submit a lab late?"
reply, found = answer(test_question)
print(f"\nQuestion: {test_question}\nAnswer: {reply}\n\nRetrieved passages:")
for c in found:
    print(f"   [page {c['page']}, similarity {c['score']:.2f}] {c['text'][:90]}...")

Loaded **Sample CS 101 syllabus**: 4 pages, 4 chunks.

Question: What happens if I submit a lab late?
Answer: Late submissions lose 10% per day for up to a maximum of 3 days, after which they receive a score of zero (p. 2). However, each student is provided two grace days per semester that can be used to submit any lab late without penalty (p. 2).

Retrieved passages:
   [page 2, similarity 0.53] Grading. Weekly lab exercises: 20%. Two mini-projects: 30% (15% each). Midterm exam: 20%. ...
   [page 4, similarity 0.33] Policies. Attendance: students must attend at least 75% of lectures to be eligible for the...
   [page 3, similarity 0.28] Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and lo...


## Step 5: Launch the application
Run this cell to open the application below the cell and create a **public link** (`https://....gradio.live`). Re-run it after changing any cell above.

In [40]:
```python
import gradio as gr
import os

gr.close_all()

APP_TITLE = "CampusGuide AI"
APP_DESCRIPTION = (
    "Your intelligent SSIT college assistant. "
    "Ask questions and explore answers backed by college documents."
)

CUSTOM_CSS = """
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
}
#main-title {
    text-align: center;
    padding: 24px 10px 12px;
    border-radius: 18px;
    background: linear-gradient(120deg, #172554, #4338ca, #7c3aed);
    color: white;
    margin-bottom: 18px;
}
#main-title h1 {
    color: white !important;
    font-size: 34px !important;
}
#main-title p {
    color: #e0e7ff !important;
    font-size: 15px !important;
}
.panel {
    border-radius: 16px !important;
}
"""

def on_upload(file):
    if file is None:
        return "No document selected."
    try:
        result = build_knowledge_base(
            read_pdf(file),
            os.path.basename(file)
        )
        return "✅ " + result
    except Exception as e:
        return f"❌ Could not read the PDF: {e}"

def use_sample():
    result = build_knowledge_base(
        SAMPLE_PAGES,
        "Sample CS 101 syllabus"
    )
    return "✅ " + result

def on_ask(question):
    if not question or not question.strip():
        raise gr.Error("Please enter a question.")

    if knowledge_base["vectors"] is None:
        raise gr.Error("Upload a PDF or select the sample document first.")

    reply, found = answer(question)

    if found:
        sources = "\n\n".join(
            f"### 📄 Page {c['page']} · Similarity {c['score']:.2f}\n"
            f"> {c['text']}"
            for c in found
        )
    else:
        sources = "No supporting passages were retrieved."

    return reply, sources

with gr.Blocks(
    title=APP_TITLE,
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="violet"
    ),
    css=CUSTOM_CSS
) as demo:

    gr.HTML("""
    <div id="main-title">
        <h1>🎓 CampusGuide AI</h1>
        <p>SSIT Smart College Assistant · Ask. Learn. Verify.</p>
    </div>
    """)

    with gr.Tabs():

        with gr.Tab("🏠 Home"):
            gr.Markdown("## Welcome to CampusGuide AI")
            gr.Markdown(
                "Your AI assistant for exploring college documents "
                "and finding information quickly."
            )

            with gr.Row():
                with gr.Column():
                    gr.Markdown("""
                    ### 📚 College Knowledge
                    Upload a college PDF to make its contents searchable.
                    """)
                with gr.Column():
                    gr.Markdown("""
                    ### 💬 Smart Questions
                    Ask questions in simple English.
                    """)
                with gr.Column():
                    gr.Markdown("""
                    ### 🔎 Source Evidence
                    Inspect the passages and page references behind answers.
                    """)

            gr.Markdown(
                "**Getting started:** Open College Documents, upload a PDF, "
                "then visit Ask CampusGuide."
            )

        with gr.Tab("📚 College Documents"):
            gr.Markdown("## College Knowledge Centre")
            gr.Markdown(
                "Upload a PDF. The current version processes one document "
                "at a time."
            )

            pdf_input = gr.File(
                label="Upload a college PDF",
                file_types=[".pdf"],
                type="filepath"
            )

            upload_status = gr.Markdown(
                f"**Current document:** {knowledge_base['name'] or 'None'}"
            )

            upload_btn = gr.Button(
                "📥 Load Document",
                variant="primary"
            )

            sample_btn = gr.Button("🧪 Try Sample Document")

            pdf_input.upload(
                on_upload,
                inputs=pdf_input,
                outputs=upload_status
            )

            upload_btn.click(
                on_upload,
                inputs=pdf_input,
                outputs=upload_status
            )

            sample_btn.click(
                use_sample,
                outputs=upload_status
            )

        with gr.Tab("💬 Ask CampusGuide"):
            gr.Markdown("## Ask Your College Assistant")

            question_box = gr.Textbox(
                label="Your question",
                placeholder="Ask something about your uploaded document...",
                lines=2
            )

            gr.Examples(
                examples=[
                    "What topics are covered in Unit V of Natural Language Processing?",
                    "What is the subject code for Deep Learning?",
                    "What are the course objectives?",
                    "What is the bus fee?"
                ],
                inputs=question_box
            )

            ask_btn = gr.Button(
                "✨ Get Answer",
                variant="primary"
            )

            answer_md = gr.Markdown(
                label="AI Answer",
                elem_classes=["panel"]
            )

            ask_btn.click(
                on_ask,
                inputs=question_box,
                outputs=answer_md
            )

            question_box.submit(
                on_ask,
                inputs=question_box,
                outputs=answer_md
            )

        with gr.Tab("📖 Verified Answers"):
            gr.Markdown("## Answers and References")
            gr.Markdown(
                "Answers appear in the Ask CampusGuide tab. "
                "Check the source evidence before relying on an answer."
            )

            verified_answer = gr.Markdown()
            verify_btn = gr.Button("Show Last Answer")

        with gr.Tab("📂 Source Evidence"):
            gr.Markdown("## Retrieved Source Passages")
            gr.Markdown(
                "Inspect the text retrieved from your document for a question."
            )

            sources_md = gr.Markdown()

            def ask_with_sources(question):
                reply, found = on_ask(question)

                sources = "\n\n".join(
                    f"### 📄 Page {c['page']} "
                    f"· Similarity {c['score']:.2f}\n> {c['text']}"
                    for c in found
                )

                return reply, sources

            ask_source_question = gr.Textbox(
                label="Question to investigate",
                placeholder="Enter a question to inspect its sources"
            )

            source_btn = gr.Button("🔍 Find Evidence", variant="primary")

            source_answer = gr.Markdown()

            source_btn.click(
                ask_with_sources,
                inputs=ask_source_question,
                outputs=[source_answer, sources_md]
            )

    gr.Markdown(
        "---\n"
        "🎓 **CampusGuide AI** · Built for the SSIT hackathon"
    )

demo.launch(share=True)
```


Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://28a9d806826ec02280.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Next steps
1. Re-read your selected problem statement (2A or 2B) in the Theme 2 document.
2. Replace the sample document and example questions with material for your problem statement.
3. Adapt `ANSWER_RULES`, the settings and the application to meet the requirements.
4. Measure the quality of the answers (see "Expected Deliverables" in the problem statement).
5. Attempt items from the **Good to have** section of the problem statement for additional credit.

**Pointers for common extensions:**
- **Markdown and text files (needed for 2B):** read them with `open(path, encoding="utf-8").read()` and treat each file (or each section under a heading) as one page, storing the file name for citations. Allow them in the application with `file_types=[".pdf", ".md", ".txt"]`.
- **Multiple documents:** use `gr.File(file_count="multiple")`, combine the pages of every file, and store the file name in each chunk for citations.
- **Answer without RAG (for comparison):** call `generate(question)` with no context and show the result next to the RAG answer.
- **Relevance threshold:** in `answer()`, if `found[0]["score"]` is below a chosen value, return "The answer could not be found in the document." without calling Gemma.
- **Conversation memory:** replace the question box with `gr.ChatInterface` and include recent messages in the prompt.

## Troubleshooting

| Issue | Resolution |
|---|---|
| The notebook remains in demonstration mode | Add `GOOGLE_API_KEY` in Secrets, enable **Notebook access**, and re-run the Step 2 cell. |
| "API key was rejected" | Create a new key at aistudio.google.com/apikey and update the secret. |
| "Usage limit reached" | Wait about one minute. Free keys allow a limited number of requests per minute. Team members may each create a key. |
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| "No text was found" | The PDF contains scanned images only. Use a text-based PDF. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |